# 🖼️ 이미지 한 바퀴 둘러보기
이미지는 숫자가 담긴 **배열**입니다. PyRewind 로 실행하면 배열이 바뀔 때마다 **그림이 어떻게 바뀌는지** 같이 보여요.

## 1. 빈 캔버스 만들기 — (높이, 너비, 색) 모양의 배열

In [ ]:
import numpy as np

img = np.zeros((40, 60, 3), dtype=np.uint8)   # 검은 그림
img.shape

## 2. 색칠하기 — 배열의 일부에 값을 넣으면 그 부분 색이 바뀝니다

In [ ]:
img[:, :20] = [230, 60, 60]     # 왼쪽: 빨강
img[:, 20:40] = [60, 200, 90]   # 가운데: 초록
img[:, 40:] = [60, 110, 230]    # 오른쪽: 파랑
img[15:25, 25:35] = 255         # 가운데 흰 네모

## 3. 흑백으로 바꾸기 — 색(마지막 축)의 평균

In [ ]:
gray = img.mean(axis=2)
gray.shape

## 4. 뒤집기 / 자르기 / 밝기

In [ ]:
flipped = img[:, ::-1]          # 좌우 뒤집기
crop = img[10:30, 15:45]        # 일부만 잘라내기
bright = np.clip(img.astype(int) + 80, 0, 255).astype(np.uint8)

## 5. 필터(합성곱) — CNN 이 하는 일
3×3 필터를 그림 위에서 움직이며 곱하고 더합니다. 함수 안으로 들어가 `out` 이 한 칸씩 쌓이는 걸 보세요. 세로 경계가 있는 곳에서 값이 커져요.

In [ ]:
kernel = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]])

def convolve(image, k):
    h, w = image.shape
    out = np.zeros((h - 2, w - 2))
    for dy in range(3):            # 필터의 9칸을 하나씩: 그림을 그만큼 밀어서 곱한 뒤 더한다
        for dx in range(3):
            out += k[dy, dx] * image[dy:dy + h - 2, dx:dx + w - 2]
    return out

edges = convolve(gray, kernel)

## 6. 여러 장을 한 번에 — (장 수, 높이, 너비)
변수 칸의 슬라이더로 한 장씩 넘겨 보세요.

In [ ]:
yy, xx = np.mgrid[:28, :28]
circles = np.stack([((xx - 14) ** 2 + (yy - 14) ** 2 < r ** 2) for r in range(3, 14, 2)]).astype(np.float32)
circles.shape

## 7. PIL 로 그림 그리기

In [ ]:
from PIL import Image, ImageDraw

pic = Image.fromarray(img)
draw = ImageDraw.Draw(pic)
draw.ellipse([4, 4, 22, 22], fill=(255, 220, 0))
small = pic.resize((30, 20))
pic

## 8. 3D 그래프 (plotly) — 출력 칸에서 마우스로 돌려 보세요
처음 한 번은 인터넷에서 plotly.js 를 불러옵니다.

In [ ]:
import plotly.graph_objects as go

x, y = np.meshgrid(np.linspace(-2, 2, 30), np.linspace(-2, 2, 30))
z = np.exp(-(x ** 2 + y ** 2))
fig = go.Figure(go.Surface(x=x, y=y, z=z))
fig.update_layout(title="가우시안 언덕")
fig.show()